# Panel de Control de Experimentos Reales (SAREnv + MTS)

Este cuaderno contiene el flujo completo y documentado para la **Fase de Integración (Middleware)** de tu TFM. 
Permite transformar mapas probabilísticos reales de **SAREnv** (matriz `.npy` y metadatos `.geojson`) a escenarios compatibles con el simulador **MTS**, realizando de forma automática la **proyección y conversión de coordenadas globales (Lat/Lon) a la rejilla local (X, Y)** de la simulación.

## Estructura del Flujo:
1. **Conversión y Middleware:** Llamada a `generar_json_real.py` para calcular el píxel de inicio $(X, Y)$ del agente a partir del LKP/IPP real de SAREnv.
2. **Configuración Dinámica:** Modificación de parámetros de simulación (número de agentes, algoritmo, batería, posición del objetivo).
3. **Ejecución del Algoritmo:** Simulación de búsqueda en el framework MTS.
4. **Análisis de Resultados:** Visualización de métricas de rendimiento (distancia, pasos tomados, éxito).

In [28]:
import os
import json
import subprocess
import pandas as pd
import numpy as np

# Cambiamos al directorio raíz del framework MTS para que las rutas relativas funcionen
current_path = os.getcwd()
if 'MTS-UncertainEnvironment-Algoritmos-bioinspirados' in current_path:
    while not os.path.exists('bf-busqueda.py'):
        os.chdir('..')

print(f"Directorio de trabajo actual: {os.getcwd()}")

Directorio de trabajo actual: c:\Users\juanc\Desktop\TFM\TFM-Juan Carlos\Software\framwork-MTS\MTS-UncertainEnvironment-Algoritmos-bioinspirados


## 1. Llamada al Middleware y Conversión de Coordenadas
Ejecutamos el script `generar_json_real.py` pasándole las rutas del GeoJSON y del Heatmap de SAREnv. El script calculará automáticamente la posición del LKP/IPP en la cuadrícula local.

In [29]:
# Rutas a los archivos exportados por SAREnv (modificar si usas otro mapa)
geojson_path = "../../SAREnv/TFM_JC/resultados/casa_de_campo/features.geojson"
npy_path = "../../SAREnv/TFM_JC/resultados/casa_de_campo/heatmap.npy"

# Ruta de salida para la configuración del escenario en MTS
config_out = "TFM_JC/pruebas/escenario_casacampo_real.json"

print("Ejecutando middleware de conversión de coordenadas y mapa...")

cmd = [
    "python", 
    "TFM_JC/scripts/generar_json_real.py",
    "--geojson", geojson_path,
    "--npy", npy_path,
    "--out", config_out,
    "--agents", "1", # Configuración por defecto: 1 agente
    "--algorithm", "voraz-heur"
]

result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8")
print(result.stdout)
if result.returncode != 0:
    print("ERROR en el middleware:")
    print(result.stderr)

Ejecutando middleware de conversión de coordenadas y mapa...
Cargando metadatos desde: ../../SAREnv/TFM_JC/resultados/casa_de_campo/features.geojson
Cargando matriz de probabilidad desde: ../../SAREnv/TFM_JC/resultados/casa_de_campo/heatmap.npy
Dimensiones del mapa leídas de .npy: 240 columnas (X) x 229 filas (Y)
Punto de inicio geográfico (LKP/IPP): Lon=-3.756782, Lat=40.422104
Zona UTM determinada: EPSG:32630
Coordenadas del centro en metros (UTM): X=435800.52, Y=4474883.23
Esquina inferior izquierda (bounds minx, miny): X=433745.60, Y=4472668.54
Posición calculada del LKP/IPP en la rejilla local de MTS: [102, 110]
Usando posiciones iniciales generadas desde LKP/IPP: [[102, 110]]
Copiando matriz .npy al directorio de destino: c:\Users\juanc\Desktop\TFM\TFM-Juan Carlos\Software\framwork-MTS\MTS-UncertainEnvironment-Algoritmos-bioinspirados\TFM_JC\pruebas\escenario_casacampo_real.npy
Punto de máxima probabilidad en la matriz (índice global): [103, 132] (Probabilidad: 1.685565e-04)
Arch

## 2. Configuración Dinámica de los Parámetros del Experimento

Ajusta aquí los parámetros interactivos para tu simulación (algoritmo, número de agentes/drones, batería de pasos, etc.).

### Modos de Posicionamiento del Objetivo (Víctima):
*   **MODO ALEATORIO (Recomendado para simulación real):** Establece `goal_pos = None`. La víctima aparecerá de forma aleatoria basada en la distribución de probabilidad real (el heatmap de SAREnv). Cambia el valor de `semilla` (ej. 0, 1, 2, 3...) para generar diferentes posiciones de la víctima y trayectorias.
*   **MODO FIJO (Recomendado para benchmarking):** Establece `goal_pos = [X, Y]` (por ejemplo, `[103, 132]` para el Zoo de Madrid). La víctima se colocará siempre en esa coordenada fija y la semilla no afectará su posición, permitiendo comparar el comportamiento de diferentes algoritmos sobre el mismo objetivo.

In [30]:
# ==============================================================================
# 2.1. PARÁMETROS GENERALES Y ALGORITMO
# ==============================================================================
algoritmo = "BHA"      # Opciones: voraz-heur, voraz-myope, ACO, ABC, BHA, lawnmower, expanding_sq
num_drones = 1                # Número de agentes/drones
bateria_pasos = 1000          # Batería/Pasos máximos de la simulación
semilla = 42                   # Cambia este número (0, 1, 2...) para generar diferentes spawns aleatorios y rutas

# ==============================================================================
# 2.2. MODO DE POSICIONAMIENTO DEL OBJETIVO (VÍCTIMA)
# ==============================================================================
# Elige una de las dos opciones (descomenta la que prefieras usar):

# OPCIÓN A: MODO ALEATORIO (Muestreo del Heatmap real usando la semilla)
goal_pos = None

# OPCIÓN B: MODO FIJO (Ubicación concreta, ej: [103, 132] es la zona del Zoo aprox.)
# goal_pos = [103, 132]

# ==============================================================================
# 2.3. PARÁMETROS DEL SENSOR
# ==============================================================================
pdmax = 0.8                   # Probabilidad máxima de detección del sensor
dmax = 2.1                    # Rango máximo de detección del sensor (en celdas)
sigma = 0.7                   # Caída exponencial de sensibilidad del sensor

# ==============================================================================
# INYECCIÓN Y GUARDADO EN EL ARCHIVO JSON
# ==============================================================================
with open(config_out, "r", encoding="utf-8") as f:
    data = json.load(f)

# Inyectar parámetros en el JSON
data["algoritmo_busqueda"] = algoritmo
data["num_agents"] = num_drones
data["num_steps"] = bateria_pasos
data["pdmax"] = pdmax
data["dmax"] = dmax
data["sigma"] = sigma
data["semilla"] = semilla

if goal_pos is None:
    data["obj_pos"] = [None]
else:
    data["obj_pos"] = goal_pos

# Si hay más de un agente, reposicionamos en diagonal para evitar colisiones iniciales
if num_drones > 1:
    lkp_x, lkp_y = data["init_pos"][0]
    data["init_pos"] = [[lkp_x + i*2, lkp_y + i*2] for i in range(num_drones)]

with open(config_out, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)

print(f"Configuración guardada en: {config_out}")
if goal_pos is None:
    print(f"Modo: ALEATORIO | Semilla: {semilla}")
else:
    print(f"Modo: FIJO | Posición Objetivo: {goal_pos}")

Configuración guardada en: TFM_JC/pruebas/escenario_casacampo_real.json
Modo: ALEATORIO | Semilla: 42


## 3. Ejecución de la Simulación en MTS
Lanzamos el simulador con la configuración que acabamos de guardar. Esto abrirá la simulación interactiva 3D en tu navegador y guardará los resultados en CSV.

In [33]:
print(f"Iniciando simulación de búsqueda con {algoritmo}...")

result = subprocess.run(["python", "bf-busqueda.py", config_out], capture_output=True, text=True, encoding="utf-8")
print(result.stdout)
if result.returncode != 0:
    print("ERROR en la simulación:")
    print(result.stderr)

Iniciando simulación de búsqueda con BHA...
Cargando mapa real desde: TFM_JC/pruebas\escenario_casacampo_real.npy
Evolución guardada en: TFG_Romeo\resultados\funciones_obj\BHA_evolution_min_agents1_iter10_FOET_20260613_210555.csv
Resultados guardados en: TFM_JC\resultados\escenario_casacampo_real\bf_bha_ET-1.csv
OK



## 4. Métricas de Rendimiento
Cargamos el archivo CSV de la última simulación para ver los resultados estadísticos.

In [34]:
import glob

nombre_prueba = os.path.basename(config_out).replace(".json", "")
search_pattern = f"**/resultados/**/{nombre_prueba}/*.csv"
csv_files = glob.glob(search_pattern, recursive=True)

if csv_files:
    latest_csv = max(csv_files, key=os.path.getmtime)
    print(f"Cargando resultados de: {latest_csv}")
    df = pd.read_csv(latest_csv, index_col=0)
    
    # Mostrar tabla estructurada
    display(df.style.set_caption("Métricas de la última simulación real"))
    
    # Resumen rápido
    if not pd.isna(df.loc["Found Target", "Target"]):
        agente_exito = int(float(df.loc["Found Target", "Target"]))
        pasos = int(float(df.loc["Steps taken", "Agent 0"]))
        print(f"\033[92m¡ÉXITO! La víctima fue localizada por el Agente {agente_exito} en {pasos} pasos.\033[0m")
    else:
        print("\033[91mFALLO: El dron agotó la batería sin localizar a la víctima.\033[0m")
else:
    print("No se encontraron archivos de resultados CSV para esta ejecución.")

Cargando resultados de: TFM_JC\resultados\escenario_casacampo_real\bf_bha_ET-1.csv


,Agent 0,Target
Initial position,[102. 110.],[105 91]
Final position,[103. 93.],[105 91]
Distance,1026.6976,0.0
Steps taken,849,0
Found Target,True,0
Semilla,-,42


¡ÉXITO! La víctima fue localizada por el Agente 0 en 849 pasos.
